In [5]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

con = duckdb.connect("./hr_data/database.duckdb")
df = con.execute("SELECT * FROM timeseries WHERE workoutId LIKE '03-06-2025_184554'").fetchdf()
con.close()

# Get the first and last X values (Time)
first_time = df['Time'].iloc[0]
last_time = df['Time'].iloc[-1]
max_idx = df['HR (bpm)'].idxmax()
max_time = df.loc[max_idx, 'Time']
max_hr = df.loc[max_idx, 'HR (bpm)']

# Plot using Plotly
fig = px.line(df, y='HR (bpm)', x='Time', title='Heart Rate')

# Get the minimum HR value
min_hr = df['HR (bpm)'].min()

# Load and sort CSV by HR ascending
df = pd.read_csv('./hr_data/zones.csv')
df = df.sort_values('HR').reset_index(drop=True)

# Build background shapes and annotations
shapes = []
annotations = []
previous_hr = min_hr - 5

colors = ['lightgray', 'green', 'lightblue', 'yellow', 'lightcoral', 'red', 'purple']
# Truncate the colors list to the number of rows in the zones df
colors = colors[:len(df)]

for i, row in df.iterrows():
    y0 = previous_hr
    y1 = row['HR']
    color = colors[i]

    # Background rectangle
    shapes.append(dict(
        type="rect",
        xref="paper", yref="y",
        x0=0, x1=1,
        y0=y0, y1=y1,
        fillcolor=color,
        opacity=0.5,
        layer="below",
        line_width=0,
    ))

    previous_hr = y1

# Update layout with shapes and annotations
fig.update_layout(shapes=shapes, annotations=annotations)

# Hover mode - the vertical line that follows the cursor
fig.update_layout(
    hovermode='x',  # or 'x unified' for grouped tooltip
    xaxis=dict(
        showspikes=True,
        spikemode='across',
        spikesnap='cursor',
        showline=True,
        spikethickness=1,
        spikecolor="gray",
        spikedash="solid",
        tickvals=[first_time, last_time],
        ticktext=[str(first_time), str(last_time)],
    ),
    yaxis=dict(
        dtick=10  # Show ticks every 10 bpm
    )
)

# Optional: consistent hover label
fig.update_traces(
    hovertemplate='Time: %{x}<br>HR: %{y} bpm',
    mode='lines'  # adds points for better hover visibility
)

#Step 3: Add marker for max point
fig.add_trace(go.Scatter(
    x=[max_time],
    y=[max_hr],
    mode='markers+text',
    marker=dict(color='red', size=10, symbol='circle'),
    text=[f'Max: {max_hr} bpm'],
    textposition='top center',
    showlegend=False
))

# Convert to a dict for Zones 1 through N
zone_colors = {i + 1: color for i, color in enumerate(colors)}

for zone, color in zone_colors.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None],  # No actual data
        mode='markers',
        marker=dict(size=10),
        name=f'Zone {zone}',
        legendgroup=f'Zone {zone}',
        showlegend=True,
        marker_color=color
    ))

fig.show()

In [8]:
con = duckdb.connect("./hr_data/database.duckdb")
hr_df = con.execute("SELECT * FROM timeseries").fetchdf()
con.close()

# 1.Get the minimum HR value
min_hr = hr_df['HR (bpm)'].min()

# 3. Build zone intervals: each zone is between previous HR and current HR
zone_bounds = []
previous_hr = min_hr-5

zones_df = pd.read_csv('./hr_data/zones.csv')
zones_df = zones_df.sort_values('HR').reset_index(drop=True)

for i, row in zones_df.iterrows():
    zone_bounds.append((previous_hr, row['HR'], row['Zone']))
    previous_hr = row['HR']

# 3. Classify each HR value into a zone
def classify_zone(hr_value):
    for lower, upper, zone in zone_bounds:
        if lower <= hr_value < upper:
            return zone
    return zone_bounds[-1][2]  # Assign to last zone if HR >= max

hr_df['Zone'] = hr_df['HR (bpm)'].apply(classify_zone)

# 4. Calculate percentage of time in each zone
zone_counts = hr_df['Zone'].value_counts().sort_index()
zone_percentages = (zone_counts / len(hr_df) * 100).round(2)
# Just to validate the DF here
# display(zone_percentages)

# 5. Display result
print("Percentage of time in each HR zone:")
pie_df = zone_percentages.to_frame(name="Percentage (%)")

# The reset_index moves the index to a column
colors = ['lightgray', 'darkgreen', 'lightblue', 'yellow', 'lightcoral']
pie_df = pie_df.reset_index()
# Just to validate the DF here

pie_df['Zone'] = pie_df.apply(
    lambda row: f"{int(row['Zone'])}: {row['Percentage (%)']:.1f}%", axis=1
)
display(pie_df)

fig = px.pie(
    pie_df,
    values='Percentage (%)',
    names='Zone',
    title='Time Spent in Each HR Zone',
    hole=0.4,  # Optional: donut chart
    color_discrete_sequence=colors[:len(pie_df)],
    category_orders={'Zone': pie_df['Zone'].tolist()}
)
fig.show()

Percentage of time in each HR zone:


,Zone,Percentage (%)
0,1: 58.6%,58.59
1,2: 11.4%,11.37
2,3: 18.4%,18.40
3,4: 6.0%,5.98
4,5: 5.7%,5.66
